# 06 — Tool-Policy Model + ReAct Loop

This notebook:
1) Trains a **tool-policy** adapter that predicts a single tool call as JSON.
2) Runs a minimal **ReAct-style loop**: tool selection → tool execution → final answer generation (with citations) + tool trace JSON.

Artifacts:
- Tool-policy adapter saved to `models/tool_policy/`
- ReAct demo uses `models/generator_dpo/`, `models/retriever/`, `models/reranker/`, and FAISS index under `data/kb/`.


In [ ]:
import os
import sys
from pathlib import Path

print('Python:', sys.version)
print('CWD:', os.getcwd())
ROOT = Path('.').resolve()

needed = [
    ROOT / 'data' / 'synthetic_tool_labels' / 'tool_train.jsonl',
    ROOT / 'models' / 'generator_dpo',
    ROOT / 'models' / 'retriever',
    ROOT / 'models' / 'reranker',
    ROOT / 'data' / 'kb' / 'faiss.index',
    ROOT / 'data' / 'kb' / 'docid_map.pkl',
    ROOT / 'data' / 'kb' / 'policies.json',
]

for p in needed:
    print(str(p), 'exists:', p.exists())


## Train tool-policy adapter

This is a lightweight run; start with 1 epoch.


In [ ]:
!python -u src/tool_policy/train_tool_policy.py \
  --train data/synthetic_tool_labels/tool_train.jsonl \
  --out models/tool_policy \
  --epochs 1 \
  --batch_size 4 \
  --grad_accum 8 \
  --lr 2e-4 \
  --max_length 512 \
  --bf16

In [ ]:
from pathlib import Path
out_dir = Path('models/tool_policy')
print('out_dir exists:', out_dir.exists())
if out_dir.exists():
    for p in sorted(out_dir.glob('*'))[:60]:
        print(p.name)


## Run the ReAct loop (demo)

This runs one tool call maximum (single-step ReAct) and prints:
- final answer (with citations)
- `TOOL_TRACE_JSON`


In [ ]:
!python -u src/tool_policy/react_loop.py \
  --question "What is your return policy for unopened items?" \
  --tool_policy_adapter models/tool_policy \
  --generator_adapter models/generator_dpo \
  --top_k 5 \
  --max_new_tokens 220

⏸️ TRAINING CHECKPOINT

Before proceeding to evaluation + serving, confirm:
- `models/tool_policy/` contains adapter weights + config + tokenizer
- ReAct demo runs end-to-end and prints `=== TOOL_TRACE_JSON ===`

Reply with:
- `ls -la models/tool_policy` (Lightning AI)
- the ReAct demo output snippet (final answer + tool trace JSON).